# P2 CHAIN C — staff directory (OOP → mongo → Flask → sockets)

Closed-book mega-question / niche hedge ladder. Reference voice: `POOP.ipynb`, `WEB DEV.ipynb` mongo misc, `SOCKETS.ipynb`.

**Requires** local MongoDB at `mongodb://localhost:27017/`. If mongo is down, still complete **C1** and **C4**; mark C2–C3 soft-fail in your session log.

**Rules**
- Dependency: C2 migrates objects from C1; C3 reads the mongo collection; C4 is the notify spine.
- Templates: `templates/chain_c/` — endpoints `desk`, `matches_view`, `hire`. Flask on port **5002**.
- **Checkpoints** are encapsulated at the **bottom** — run after attempting. Not a writing guide.
- **No answer keys in this notebook.** Archive answers only when marking.
- Prefer not to leave mongo collections / local junk lying around after a session (`chain_c_db.staff` can be dropped when done).


---
## Part C1 — in-memory staff collection (10 min)

`Staff(staff_id, surname, given, department)` with privates + getters + `as_dict()` for mongo.

`Directory`: `add(staff)`, `find_by_id(staff_id)`, `all_dicts()` → list of dicts.

Populate `roster = Directory()` with ≥5 staff. Print `all_dicts()`.


In [3]:
# Part C1 — YOUR ATTEMPT

roster = None

class Staff:
    def __init__(self, staff_id, surname, given, department):
        self.__staff_id = staff_id
        self.__surname = surname
        self.__given = given
        self.__department = department
        
    def get_staff_id(self):
        return self.__staff_id
    
    def set_staff_id(self, data):
        self.__staff_id = data
        
    def get_surname(self):
        return self.__surname
    
    def set_surname(self, data):
        self.__surname = data    
        
    def get_given(self):
        return self.__given
    
    def set_given(self, data):
        self.__given = data   
        
    def get_department(self):
        return self.__department
    
    def set_department(self, data):
        self.__department = data
        
    def as_dict(self):
        return {'staff_id': self.get_staff_id(), 'surname': self.get_surname(), 'given': self.get_given(), 'department': self.get_department()}
    
class Directory:
    def __init__(self):
        self.__arr = []
        
    def add(self, new):
        self.__arr.append(new)
    
    def find_by_id(self, sid):
        for item in self.__arr:
            if item.get_id() == sid:
                return item
        else: 
            return None
        
    def all_dicts(self):
        final = []
        for item in self.__arr:
            final.append(item.as_dict())
        return final
    
roster = Directory()
roster.add(Staff('00', 'ayanami', 'rei','Biology'))
roster.add(Staff('01', 'ikari', 'shinji', 'Language Arts'))
roster.add(Staff('02', 'soryu', 'asuka langley', 'Biology'))
roster.add(Staff('03', 'suzuhara', 'toji', 'PE&CCA'))
roster.add(Staff('04', 'uzumaki', 'naruto', 'Principal'))
roster.add(Staff('05', 'kagenou', 'cid', 'Economics'))
roster.all_dicts()

[{'staff_id': '00',
  'surname': 'ayanami',
  'given': 'rei',
  'department': 'Biology'},
 {'staff_id': '01',
  'surname': 'ikari',
  'given': 'shinji',
  'department': 'Language Arts'},
 {'staff_id': '02',
  'surname': 'soryu',
  'given': 'asuka langley',
  'department': 'Biology'},
 {'staff_id': '03',
  'surname': 'suzuhara',
  'given': 'toji',
  'department': 'PE&CCA'},
 {'staff_id': '04',
  'surname': 'uzumaki',
  'given': 'naruto',
  'department': 'Principal'},
 {'staff_id': '05',
  'surname': 'kagenou',
  'given': 'cid',
  'department': 'Economics'}]

---
## Part C2 — migrate to mongo (12 min)

`MongoClient("mongodb://localhost:27017/")`:

- DB `chain_c_db`, collection `staff`
- `drop()` for a repeatable run
- `insert_many(roster.all_dicts())`
- `find` with inclusion projection on `surname`, `department`
- `update_one` one staff department with `{"$set": {...}}` — **filter first**

Keep `coll` for C3.


In [16]:
# Part C2 — YOUR ATTEMPT
from pymongo import MongoClient

coll = None

#connect
client = MongoClient("mongodb://localhost:27017/")
db = client['chain_c_db']
coll = db['staff']

#seed
coll.delete_many({})
coll.insert_many(roster.all_dicts())

for row in coll.find({'department': 'Biology'},{'_id':0, 'surname': 1, 'department': 1}):
    print(row)

coll.update_one({'staff_id': '05'},{'$set': {'staff_id': '05', 'surname': 'white', 'given': 'walter', 'department': 'Chemistry'}})


{'surname': 'ayanami', 'department': 'Biology'}
{'surname': 'soryu', 'department': 'Biology'}


---
## Part C3 — Flask over mongo (18 min)

Templates in `templates/chain_c/`:

| Route | Endpoint fn | Template | Notes |
|-------|-------------|----------|-------|
| `/` GET+POST | `desk` | `chain_c/desk.html` | form keys `dept_query`, `surname_query` → redirect |
| `/matches/<dept_query>/<surname_query>` | `matches_view` | `chain_c/matches.html` | mongo find; pass `people` |
| `/hire` GET+POST | `hire` | `chain_c/hire.html` | keys `sid`, `surname`, `given`, `department`; `insert_one`; redirect `desk` |

`get_mongo()` returns the `staff` collection. Port **5002**.


In [ ]:
# Part C3 — YOUR ATTEMPT
from flask import Flask, render_template, request, redirect, url_for
# from pymongo import MongoClient

app = Flask(__name__)

accept = ['GET', 'POST']

@app.route("/", methods = accept)
def desk():
    error = None
    if request.method == 'POST':
        dept_query = request.form.get('dept_query','')
        surname_query = request.form.get('surname_query','')
        if dept_query and surname_query:
            return redirect(url_for('chain_c/matches.html'), dept_query = dept_query, surname_query = surname_query)
        error = 'Please fill in all fields'
    return render_template(url_for('chain_c/desk.html'), error = error)

@app.route("/matches/<dept_query>/<surname_query>", methods = accept)
def matches(dept_query, surname_query):
    return render_template(
        url_for('chain_c/matches.html'), 
        people = coll.find(
            {'department': dept_query, 'surname': surname_query},
            {'_id': 0, 'surname':1,'given':1,'department':1,'staff_if':0}
        )
    )

@app.route("/hire")
def hire():
    error = None
    if request.method == 'POST':
        sid = request.form.get('sid','')
        surname = request.form.get('surname','')
        given = request.form.get('given','')
        department = request.form.get('department','')
        coll.insert_one({'staff_id': sid, 'surname': surname, 'given': given, 'department': department})
        if sid and surname and given and department:
            redirect(url_for('chain_c/desk.html'))
        error = 'Please fill in all fields'
    return render_template(url_for('chain_c/hire.html'), error = error)

---
## Part C4 — sockets notify spine (12 min)

1. `recv_line(sock)` loops `recv(1024)` until newline (or peer closed), decode utf-8, strip.
2. Comment the **server** spine: socket → bind → listen → accept → loop recv_line / sendall → close
   and **client** spine: connect → sendall `(line+"\n").encode()` → recv → close.
3. Bonus: client sends `UPDATED <staff_id>`; server prints it.

Prefer `sendall`; always encode/decode utf-8.


In [ ]:
# Part C4 — YOUR ATTEMPT

def recv_line(sock):
    

# optional mini server/client below


---
# Checkpoints (encapsulated)

Run **after** attempting. Edit `PARTS` to choose what to mark.


In [ ]:
# CHECKPOINTS — Chain C (run after attempts)

PARTS = ("C1", "C2", "C3", "C4")


def _check_c1():
    if roster is None or len(roster.all_dicts()) < 5:
        return False, "roster needs >= 5 staff"
    d = roster.all_dicts()[0]
    for k in ("staff_id", "surname", "given", "department"):
        if k not in d:
            return False, f"as_dict missing {k}"
    return True, "ok"


def _check_c2():
    if coll is None:
        return False, "coll is None — mongo migrate not done"
    n = coll.count_documents({})
    if n < 5:
        return False, f"count {n} < 5"
    return True, "ok"


def _check_c3():
    for name in ("app", "get_mongo", "desk", "matches_view", "hire"):
        if name not in globals():
            return False, f"missing {name}"
    eps = {rule.endpoint for rule in app.url_map.iter_rules()}
    for ep in ("desk", "matches_view", "hire"):
        if ep not in eps:
            return False, f"endpoint {ep} not registered"
    return True, "ok (manual: app.run port 5002)"


def _check_c4():
    if "recv_line" not in globals() or not callable(recv_line):
        return False, "recv_line missing or not callable"
    return True, "ok"


_CHECKS = {"C1": _check_c1, "C2": _check_c2, "C3": _check_c3, "C4": _check_c4}

for part in PARTS:
    try:
        ok, detail = _CHECKS[part]()
    except Exception as e:
        ok, detail = False, str(e) + (" (is mongod running?)" if part == "C2" else "")
    print(f"{part}: {'PASS' if ok else 'FAIL — ' + detail}")
